# 04 — Validation and historical analysis examples
    Optional, read-only checks after 03. Not a scheduled training notebook.
    The fixed historical example does not account for NOAA publication delay and must not be used
    unchanged as a prediction-feature join. An unmatched hour stays missing; no rain is not assumed.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Uniqueness, expected-hour coverage, and unmapped bike IDs

In [0]:
from pyspark.sql import functions as F
mapping = spark.table("workspace.default.citibike_noaa_mapping")
hourly = spark.table("workspace.default.nyc_weather_hourly_with_precipitation")
if mapping.limit(1).count() == 0:
    raise ValueError("Mapping is empty.")
if mapping.groupBy("station_id").count().filter("count > 1").limit(1).count():
    raise ValueError("More than one mapping per bike station.")
if hourly.groupBy("noaa_station_id", "hour_start_utc").count().filter("count > 1").limit(1).count():
    raise ValueError("Duplicate station-hour rows.")
display(mapping.groupBy("noaa_station_id").count().orderBy("noaa_station_id"))

# Distinguish truly absent hours from unknown precipitation in represented hours.
display(spark.sql("""
WITH stations AS (
 SELECT DISTINCT noaa_station_id FROM workspace.default.citibike_noaa_mapping
), hours AS (
 SELECT explode(sequence(TIMESTAMP '2026-08-21 00:00:00',
                         TIMESTAMP '2026-10-25 23:00:00', INTERVAL 1 HOUR)) AS hour_start_utc
)
SELECT s.noaa_station_id, COUNT(*) AS expected_hours,
       SUM(CASE WHEN w.noaa_station_id IS NULL THEN 1 ELSE 0 END) AS absent_hours,
       SUM(CASE WHEN w.noaa_station_id IS NOT NULL AND w.precipitation_interpretation = 'unknown'
                THEN 1 ELSE 0 END) AS represented_unknown_precip_hours
FROM stations s CROSS JOIN hours h
LEFT JOIN workspace.default.nyc_weather_hourly_with_precipitation w
 ON s.noaa_station_id = w.noaa_station_id AND h.hour_start_utc = w.hour_start_utc
GROUP BY s.noaa_station_id ORDER BY s.noaa_station_id
"""))

# Historical status IDs lacking a current mapping; do not assume the difference is exactly five.
display(spark.sql("""
SELECT DISTINCT s.station_id
FROM workspace.default.silver_station_status s
LEFT ANTI JOIN workspace.default.citibike_noaa_mapping m
ON s.station_id = m.station_id
ORDER BY s.station_id
"""))

## Inspect original report messages

In [0]:
display(spark.sql("""
SELECT
    STATION,
    observed_at_utc,
    precipitation,
    precipitation_Measurement_Code,
    precipitation_Source_Code,
    precipitation_Report_Type,
    REM,
    REM_Source_Code
FROM workspace.default.noaa_weather_raw
WHERE STATION = 'USW00014732'
  AND observed_at_utc >= TIMESTAMP '2026-10-23 00:00:00'
  AND observed_at_utc <  TIMESTAMP '2026-10-24 00:00:00'
  AND REM IS NOT NULL
ORDER BY observed_at_utc
LIMIT 30
"""))

## Demonstrate the historical bike/weather join

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

display(spark.sql("""
WITH bike_sample AS (
    SELECT
        station_id,
        fetched_at,
        num_bikes_available,
        num_docks_available,
        is_renting,
        is_returning
    FROM workspace.default.silver_station_status
    WHERE nyc_date = DATE '2026-10-01'
      AND fetched_at >= TIMESTAMP '2026-10-01 12:00:00'
      AND fetched_at <  TIMESTAMP '2026-10-01 12:10:00'
    ORDER BY fetched_at, station_id
    LIMIT 100
)

SELECT
    b.station_id,
    b.fetched_at AS bike_observed_at_utc,
    b.num_bikes_available,
    b.num_docks_available,
    b.is_renting,
    b.is_returning,

    m.noaa_station_id,
    ROUND(m.distance_km, 2) AS weather_station_distance_km,

    w.hour_start_utc AS weather_hour_start_utc,
    w.temperature,
    w.wind_speed,
    w.precipitation_interpreted,
    w.precipitation_interpretation,
    w.precipitation_maintenance_flag,

    CASE
        WHEN m.noaa_station_id IS NULL THEN 'no_station_mapping'
        WHEN w.noaa_station_id IS NULL THEN 'no_weather_hour'
        ELSE 'matched'
    END AS join_status

FROM bike_sample b
LEFT JOIN workspace.default.citibike_noaa_mapping m
    ON b.station_id = m.station_id
LEFT JOIN workspace.default.nyc_weather_hourly_with_precipitation w
    ON m.noaa_station_id = w.noaa_station_id
   AND w.hour_start_utc =
       date_trunc('HOUR', b.fetched_at) - INTERVAL 1 HOUR

ORDER BY b.fetched_at, b.station_id
"""))

In [0]:
display(spark.sql("""
WITH unmapped AS (
    SELECT DISTINCT s.station_id
    FROM workspace.default.silver_station_status s
    LEFT ANTI JOIN workspace.default.citibike_noaa_mapping m
        ON s.station_id = m.station_id
)
SELECT
    u.station_id,
    i.name,
    i.lat,
    i.lon,
    i.snapshot_date,
    i._has_invalid_coordinates,
    i._has_parse_error,
    CASE
        WHEN i.station_id IS NULL THEN 'no_metadata_record'
        ELSE 'metadata_present_review_location'
    END AS review_status
FROM unmapped u
LEFT JOIN workspace.default.silver_station_info i
    ON u.station_id = i.station_id
ORDER BY u.station_id, i.snapshot_date DESC
"""))

In [0]:
display(spark.sql("""
    SHOW VIEWS IN workspace.default
    LIKE 'nyc_weather_hourly_with_precipitation'
"""))

In [0]:
display(spark.table(
    "workspace.default.nyc_weather_hourly_with_precipitation"
).limit(20))